# 01 - Data Preparation (Full Image)

This notebook prepares the dataset for the **first finetuning stage**.
Each image is resized and synthetic blur is applied.

> **Note on image resizing:** The original images are 1024×1024 pixels.
> Due to GPU memory constraints during training (limited VRAM),
> images are resized to 512×512 before blur synthesis.
> This reduces memory usage while preserving sufficient image detail for training.
> If you have sufficient GPU memory, you may skip the resizing step
> and use the original resolution directly.

**Run this notebook once before training.**

```
Original sharp images
        ↓
  Resize to 512×512 (GPU memory constraint)
        ↓
  Apply synthetic blur (Gaussian + Motion)
        ↓
  Save blur/sharp pairs → ready for training
```

## (Optional) Mount Google Drive

Run this cell only if you are using **Google Colab** and your dataset is stored on Google Drive.
Skip this cell if you are running locally.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Step 1 - Set Paths

Set the paths below according to your directory structure.

```
# Google Colab + Drive example:
# SHARP_ROOT = '/content/drive/MyDrive/your_project/dataset'
# OUTPUT_DIR = '/content/drive/MyDrive/your_project/dataset_resize'

# Local machine example:
# SHARP_ROOT = '/path/to/your/dataset'
# OUTPUT_DIR = '/path/to/your/dataset_resize'
```

The `SHARP_ROOT` folder must contain `train/`, `val/`, and `test/` subfolders with sharp images.

In [ ]:
# ← SET YOUR PATHS HERE
SHARP_ROOT = 'dataset_path'   # folder containing train/ val/ test/ subfolders
OUTPUT_DIR = 'output_path'    # output folder where blur/sharp pairs will be saved

# Verify folders
from pathlib import Path

for split in ['train', 'val', 'test']:
    p = Path(SHARP_ROOT) / split
    if p.exists():
        n = len([f for f in p.iterdir() if f.suffix.lower() in ('.png','.jpg','.jpeg')])
        print(f'{split:6s}: {n} images found')
    else:
        print(f'{split:6s}: NOT FOUND - {p}')

## Step 2 - Resize Images to 512×512

All images are resized before blur is applied.
This ensures consistent input size for the model.

> Resized sharp images are saved to `OUTPUT_DIR/<split>/sharp/`.

In [ ]:
import cv2
from pathlib import Path
from tqdm import tqdm

TARGET = 512  # target resolution

for split in ['train', 'val', 'test']:
    src_dir = Path(SHARP_ROOT) / split
    dst_dir = Path(OUTPUT_DIR) / split / 'sharp'
    if not src_dir.exists():
        print(f'{split}: not found, skipping.')
        continue
    dst_dir.mkdir(parents=True, exist_ok=True)
    imgs = sorted([f for f in src_dir.iterdir()
                   if f.suffix.lower() in ('.png','.jpg','.jpeg')])
    for img_path in tqdm(imgs, desc=f'Resizing {split}'):
        img = cv2.imread(str(img_path))
        if img is None: continue
        resized = cv2.resize(img, (TARGET, TARGET), interpolation=cv2.INTER_LANCZOS4)
        cv2.imwrite(str(dst_dir / img_path.name), resized)
    print(f'{split}: {len(imgs)} images resized to {TARGET}x{TARGET} → {dst_dir}')

## Step 3 - Apply Synthetic Blur

For each sharp image, a blurred version is generated and saved to `OUTPUT_DIR/<split>/blur/`.

Two types of blur are applied randomly:
- **Gaussian blur** — simulates out-of-focus / camera shake
- **Motion blur** — simulates fast-moving objects or camera motion



In [ ]:
import cv2, numpy as np, random
from pathlib import Path
from tqdm import tqdm

# --- BLUR SETTINGS ---
GAUSSIAN_SIGMA_RANGE = (1.5, 4.0)
MOTION_KERNEL_RANGE  = (7, 21)
BLUR_MIX_PROB        = 0.5   # 0.5 = 50% Gaussian, 50% Motion blur
# ----------------------

def apply_gaussian_blur(img):
    sigma = random.uniform(*GAUSSIAN_SIGMA_RANGE)
    k = int(2 * round(3 * sigma) + 1)
    if k % 2 == 0: k += 1
    return cv2.GaussianBlur(img, (k, k), sigma)

def apply_motion_blur(img):
    k = random.choice(range(MOTION_KERNEL_RANGE[0], MOTION_KERNEL_RANGE[1]+1, 2))
    angle = random.uniform(0, 360)
    kernel = np.zeros((k, k))
    kernel[k//2, :] = 1.0 / k
    M = cv2.getRotationMatrix2D((k//2, k//2), angle, 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= kernel.sum() + 1e-8
    return cv2.filter2D(img, -1, kernel)

def generate_blur(img):
    if random.random() < BLUR_MIX_PROB:
        return apply_gaussian_blur(img)
    else:
        return apply_motion_blur(img)

for split in ['train', 'val', 'test']:
    sharp_dir = Path(OUTPUT_DIR) / split / 'sharp'
    blur_dir  = Path(OUTPUT_DIR) / split / 'blur'
    if not sharp_dir.exists():
        print(f'{split}: sharp folder not found, skipping.')
        continue
    blur_dir.mkdir(parents=True, exist_ok=True)
    imgs = sorted(sharp_dir.glob('*'))
    for p in tqdm(imgs, desc=f'{split} blur'):
        img = cv2.imread(str(p))
        if img is None: continue
        cv2.imwrite(str(blur_dir / p.name), generate_blur(img))
    print(f'{split}: {len(imgs)} blur images saved → {blur_dir}')

print('All done! Sharp and blur pairs generated in output directory.')

## Dataset Ready

The dataset is now prepared with the following structure:
```
OUTPUT_DIR/
  train/
    sharp/   ← resized original images (512×512)
    blur/    ← synthetically blurred images
  val/
    sharp/
    blur/
  test/
    sharp/
    blur/
```

Proceed to **02_training_resize.ipynb** to start finetuning.